# Lesson 12a: Exploration Strategies Theory

**Learning Objectives:**
- Understand the exploration-exploitation tradeoff
- Master multi-armed bandit algorithms (UCB, Thompson Sampling)
- Learn count-based exploration methods
- Study intrinsic motivation and curiosity-driven learning
- Explore information gain and entropy-based methods
- Understand Never Give Up (NGU) and other modern approaches

**Prerequisites:** MDPs (1a), Q-Learning (4a), DQN (7a)

**References:**
- Sutton & Barto Chapter 2: Multi-armed Bandits
- Bellemare et al. (2016) - Count-based exploration
- Pathak et al. (2017) - Curiosity-driven exploration (ICM)
- Burda et al. (2018) - Random Network Distillation (RND)
- Badia et al. (2020) - Never Give Up (NGU)

## 1. The Exploration-Exploitation Tradeoff

### 1.1 Fundamental Challenge

**Exploitation**: Choose actions known to yield high reward
$$a^* = \arg\max_a Q(s,a)$$

**Exploration**: Try new actions to discover potentially better options
$$a \sim \text{UniformRandom}(\mathcal{A})$$

**Dilemma**: Must balance both to maximize long-term reward

### 1.2 Why Exploration Matters

**Without exploration:**
- Agent may miss better strategies
- Gets stuck in local optima
- Poor generalization

**Examples of exploration failure:**
1. Montezuma's Revenge: Sparse rewards require extensive exploration
2. Robotics: Must discover diverse behaviors
3. Dialogue systems: Need to explore conversation strategies

### 1.3 Types of Exploration

**Undirected exploration:**
- ε-greedy: Random action with probability ε
- Boltzmann: Sample from softmax distribution
- Action noise: Add Gaussian noise (continuous control)

**Directed exploration:**
- Optimism under uncertainty (UCB)
- Thompson sampling (Bayesian)
- Count-based bonuses
- Curiosity-driven
- Information gain

## 2. Multi-Armed Bandits

### 2.1 Bandit Problem Formulation

**Setting**: $K$ actions (arms), each with unknown reward distribution
- Pull arm $a_t$ at time $t$
- Observe reward $r_t \sim p(r|a_t)$
- Goal: Maximize $\sum_{t=1}^T r_t$

**Regret**: Difference from optimal policy
$$\text{Regret}(T) = T \cdot \max_a \mathbb{E}[r|a] - \sum_{t=1}^T r_t$$

Want: Sublinear regret $O(\log T)$

### 2.2 ε-Greedy

**Algorithm:**
$$a_t = \begin{cases}
\arg\max_a Q(a) & \text{with probability } 1-\epsilon \\
\text{random action} & \text{with probability } \epsilon
\end{cases}$$

**Update:**
$$Q(a) \leftarrow Q(a) + \alpha(r - Q(a))$$

**Properties:**
- Simple, widely used
- Explores uniformly (not directed)
- Regret: $O(T)$ with constant ε (linear!)
- Can improve with decaying ε: $\epsilon_t = 1/t$

### 2.3 Upper Confidence Bound (UCB)

**Principle**: Optimism in the face of uncertainty

**UCB1 Algorithm:**
$$a_t = \arg\max_a \left[ Q(a) + c\sqrt{\frac{\log t}{N(a)}} \right]$$

where:
- $Q(a)$: Estimated value
- $N(a)$: Number of times action $a$ selected
- $c$: Exploration coefficient
- $\sqrt{\frac{\log t}{N(a)}}$: Uncertainty bonus

**Intuition:**
- Rarely-tried actions get high bonus → explore
- As $N(a)$ increases, bonus decreases → exploit
- Logarithmic regret: $O(\log T)$ ✓

### 2.4 Thompson Sampling

**Bayesian approach**: Maintain distribution over action values

**Algorithm:**
1. Maintain posterior $p(Q(a)|\mathcal{D})$ for each action
2. Sample $\tilde{Q}(a) \sim p(Q(a)|\mathcal{D})$ for all actions
3. Select $a_t = \arg\max_a \tilde{Q}(a)$
4. Observe reward, update posterior

**For Bernoulli rewards:**
- Prior: $Q(a) \sim \text{Beta}(\alpha_a, \beta_a)$
- Update on success: $\alpha_a \leftarrow \alpha_a + 1$
- Update on failure: $\beta_a \leftarrow \beta_a + 1$

**Properties:**
- Provably optimal regret: $O(\log T)$
- Automatically balances exploration/exploitation
- More complex than UCB
- Popular in practice (A/B testing, recommendation systems)

## 3. Count-Based Exploration

### 3.1 Exploration Bonuses

**Idea**: Add bonus to reward for visiting novel states

$$r^+(s,a) = r(s,a) + \beta \cdot b(s,a)$$

where $b(s,a)$ is exploration bonus

### 3.2 Simple Count-Based Bonus

**Track visit counts:**
$$N(s,a) = \text{number of times } (s,a) \text{ visited}$$

**Exploration bonus:**
$$b(s,a) = \frac{1}{\sqrt{N(s,a)}}$$

or

$$b(s,a) = \frac{\beta}{\sqrt{N(s,a) + 1}}$$

**Properties:**
- High bonus for unvisited states
- Bonus decreases as state visited more
- Simple, effective in tabular settings
- **Problem**: Doesn't scale to high-dimensional state spaces!

### 3.3 Pseudo-Counts (Bellemare et al. 2016)

**Challenge**: Count states in high-dimensional spaces (images)

**Solution**: Learn density model $\rho(s)$, derive pseudo-count

**Density model:**
$$\rho_n(s) = \text{probability of state } s \text{ under model trained on } n \text{ samples}$$

**Recoding probability** (likelihood after seeing $s$):
$$\rho_n'(s) = \text{probability under model after adding } s$$

**Pseudo-count** $\hat{N}(s)$ satisfies:
$$\rho_n(s) = \frac{\hat{N}(s)}{n}, \quad \rho_n'(s) = \frac{\hat{N}(s) + 1}{n + 1}$$

Solving:
$$\hat{N}(s) = \frac{\rho_n(s)(1 - \rho_n'(s))}{\rho_n'(s) - \rho_n(s)}$$

**Exploration bonus:**
$$b(s) = \frac{\beta}{\sqrt{\hat{N}(s) + 0.01}}$$

**Models used:**
- CTS (Context Tree Switching) for pixel-based
- PixelCNN for Atari

### 3.4 Hash-Based Counts (SimHash)

**Idea**: Use locality-sensitive hashing to bin similar states

**Algorithm:**
1. Extract features $\phi(s)$ (e.g., from neural network)
2. Hash: $h(s) = \text{LSH}(\phi(s))$
3. Count visits: $N(h(s))$
4. Bonus: $b(s) = \beta / \sqrt{N(h(s))}$

**Benefits:**
- Fast, scalable
- Generalizes across similar states
- Used in practice (Uber's Go-Explore)

## 4. Curiosity-Driven Exploration

### 4.1 Intrinsic Motivation

**Concept**: Internal reward for learning/novelty (beyond task reward)

$$r_t^{\text{total}} = r_t^{\text{extrinsic}} + \beta \cdot r_t^{\text{intrinsic}}$$

**Types of intrinsic motivation:**
1. **Novelty**: Reward for visiting new states
2. **Learning progress**: Reward for improving predictions
3. **Empowerment**: Reward for actions that increase future control
4. **Information gain**: Reward for reducing uncertainty

### 4.2 Forward Dynamics Error (Prediction Error)

**Idea**: Surprise = intrinsic reward

Learn forward model:
$$\hat{s}_{t+1} = f_\theta(s_t, a_t)$$

Intrinsic reward:
$$r_t^{\text{int}} = \|s_{t+1} - \hat{s}_{t+1}\|^2$$

**Intuition**: High prediction error = novel/surprising state → explore!

**Problem**: "Noisy TV" problem
- Stochastic dynamics → high error forever
- Agent attracted to unpredictable events (e.g., random noise)
- Doesn't distinguish aleatoric from epistemic uncertainty

### 4.3 Intrinsic Curiosity Module (ICM)

**Pathak et al. (2017)**: Use feature space, not raw states

**Components:**
1. **Feature encoder**: $\phi: s \to z$ (learned representation)
2. **Forward model**: $\hat{z}_{t+1} = f(z_t, a_t)$
3. **Inverse model**: $\hat{a}_t = g(z_t, z_{t+1})$

**Intrinsic reward** (forward model error in feature space):
$$r_t^{\text{int}} = \frac{\eta}{2}\|\hat{z}_{t+1} - z_{t+1}\|^2$$

**Training:**
- Forward loss: $\mathcal{L}_F = \|\hat{z}_{t+1} - z_{t+1}\|^2$
- Inverse loss: $\mathcal{L}_I = -\log P(\hat{a}_t = a_t)$
- Total: $\mathcal{L} = (1-\beta)\mathcal{L}_I + \beta\mathcal{L}_F$

**Why inverse model?**
- Encourages $\phi$ to encode features relevant to agent's actions
- Filters out uncontrollable aspects (e.g., background noise)

**Benefits:**
- Mitigates "noisy TV" problem
- Works on raw pixels (Atari, VizDoom)
- State-of-the-art sparse reward performance

### 4.4 Random Network Distillation (RND)

**Burda et al. (2018)**: Compare prediction to random target network

**Networks:**
1. **Target**: $f_\theta: s \to y$ (fixed random network)
2. **Predictor**: $\hat{f}_\phi: s \to \hat{y}$ (trained to match target)

**Intrinsic reward** (prediction error):
$$r_t^{\text{int}} = \|f_\theta(s_{t+1}) - \hat{f}_\phi(s_{t+1})\|^2$$

**Training predictor:**
$$\mathcal{L} = \|f_\theta(s) - \hat{f}_\phi(s)\|^2$$

**Intuition:**
- Novel states: predictor hasn't seen → high error
- Familiar states: predictor trained → low error
- Random target prevents forgetting (unlike forward model)

**Benefits:**
- No need for dynamics model
- Handles stochastic environments better
- State-of-the-art on Montezuma's Revenge!

## 5. Information-Theoretic Exploration

### 5.1 Entropy Maximization

**Objective**: Maximize policy entropy
$$J(\pi) = \mathbb{E}_{\tau \sim \pi}\left[\sum_t r_t + \alpha H(\pi(\cdot|s_t))\right]$$

where $H(\pi(\cdot|s)) = -\sum_a \pi(a|s)\log\pi(a|s)$

**Already seen in**: Soft Actor-Critic (SAC) - Lesson 10a

**Benefits:**
- Automatic exploration
- Robustness (multi-modal policies)
- Implicit coverage of state space

### 5.2 State Entropy (Diversity)

**Maximize state visitation entropy:**
$$H(\rho^\pi) = -\int \rho^\pi(s)\log\rho^\pi(s)ds$$

where $\rho^\pi(s)$ is state visitation distribution

**Practical approximation:**
- Maintain state visit counts
- Bonus: $b(s) \propto -\log\rho(s)$

### 5.3 Information Gain

**Objective**: Select actions that reduce uncertainty

**Bayesian setting:**
- Maintain belief $p(\theta|\mathcal{D})$ over model parameters

**Information gain** about $\theta$ from $(s,a)$:
$$IG(s,a) = H(p(\theta|\mathcal{D})) - \mathbb{E}_{s'}[H(p(\theta|\mathcal{D} \cup \{s,a,s'\}))]$$

**Equivalent to**: Expected reduction in posterior entropy

**Bayesian RL algorithms:**
- BOSS (Bayesian Optimization for Sparse Sampling)
- BAMCP (Bayes-Adaptive Monte Carlo Planning)

**Challenges**: Maintaining posterior is expensive

## 6. Modern Exploration Methods

### 6.1 Never Give Up (NGU)

**Badia et al. (2020)**: Combines episodic and lifelong novelty

**Two intrinsic rewards:**

1. **Episodic novelty** (within episode):
   - Store embedding of visited states in episodic memory
   - Compute similarity to current state using k-NN
   - High novelty if dissimilar to recent states

2. **Lifelong novelty** (across episodes):
   - RND-style prediction error
   - Measures novelty across entire training

**Combined intrinsic reward:**
$$r^{\text{int}} = r^{\text{episodic}} \cdot \min(\max(r^{\text{lifelong}}, 1), L)$$

**Benefits:**
- Episodic: Avoids revisiting in same episode
- Lifelong: Avoids revisiting across episodes
- State-of-the-art on hard exploration Atari games

### 6.2 Agent57

**Badia et al. (2020)**: First agent to exceed human on all 57 Atari games

**Key innovations:**
1. **Population of policies** with different exploration/exploitation tradeoffs
2. **Meta-controller** selects which policy to use
3. **Adaptive intrinsic reward scaling** (per policy)
4. **Retrace** for off-policy learning

**Architecture:**
```
Policy family: {π_i}_i=1^N
- π_1: Pure exploitation (β=0)
- π_N: Pure exploration (β_max)
- π_i: Interpolate between

Meta-controller:
- Learns which policy works best
- Uses bandit algorithm
```

### 6.3 Go-Explore

**Ecoffet et al. (2019, 2021)**: Archive-based exploration

**Phase 1: Exploration**
1. Maintain archive of interesting states
2. Select state from archive
3. Return to that state (via demonstration or learned policy)
4. Explore from there
5. Add new interesting states to archive

**Phase 2: Robustification**
- Train robust policy from scratch
- Use trajectories discovered in Phase 1
- Imitation learning

**State representation:**
- Downsampled images
- Hash-based (group similar states)

**"Interesting" states:**
- First time visiting a cell
- Achieved high score
- Novel trajectory

**Results**: Solved Montezuma's Revenge, Pitfall (previously unsolved)

**Controversy**: Phase 1 uses deterministic resets (not realistic), but Phase 2 is fully robust

## 7. Comparative Analysis

### 7.1 Method Comparison

| Method | Type | Pros | Cons | Best For |
|--------|------|------|------|----------|
| ε-greedy | Undirected | Simple, fast | Inefficient | Simple tasks |
| UCB | Directed (count) | Optimal regret | Tabular only | Bandits |
| Thompson | Directed (Bayesian) | Optimal, principled | Complex posterior | Bandits, simple MDPs |
| Count-based | Directed (count) | Interpretable | Scales poorly | Tabular/small state |
| Pseudo-counts | Directed (density) | Scales better | Density modeling hard | Atari |
| ICM | Curiosity | Handles pixels | Can be distracted | Sparse reward |
| RND | Curiosity | Robust, simple | Hyperparameter sensitive | Very sparse reward |
| NGU | Hybrid | State-of-the-art | Complex, slow | Hard exploration |
| Go-Explore | Archive | Solves hardest tasks | Needs resets (Phase 1) | Deterministic envs |

### 7.2 When to Use What

**Dense rewards:**
- ε-greedy or entropy regularization (SAC) usually sufficient

**Sparse but reachable:**
- Count-based bonuses
- ICM if state space is large

**Very sparse (e.g., Montezuma's Revenge):**
- RND or NGU
- Go-Explore if environment allows

**Stochastic environments:**
- Avoid forward dynamics error
- Use RND, count-based, or NGU

**Continuous control:**
- Entropy regularization (SAC)
- Action noise with decay

### 7.3 Combining Methods

**Common combinations:**
1. **ε-greedy + count bonuses**: Simple but effective
2. **Entropy regularization + RND**: Used in Agent57
3. **ICM + PPO**: Popular for sparse reward tasks
4. **Multiple intrinsic rewards**: NGU approach

**Design considerations:**
- Intrinsic reward scaling: $\beta$ is crucial hyperparameter
- Reward normalization: Keep intrinsic and extrinsic comparable
- Curriculum: Start with high $\beta$, decay over time

## 8. Theoretical Foundations

### 8.1 PAC-MDP Framework

**Probably Approximately Correct in MDPs**

An algorithm is PAC-MDP if with probability $1-\delta$:
- Learns near-optimal policy: $V^\pi \geq V^* - \epsilon$
- With sample complexity polynomial in relevant quantities
- In polynomial time

**Exploration algorithms with PAC guarantees:**
- R-MAX
- UCRL2
- Model-based interval estimation (MBIE)

### 8.2 Regret Bounds

**Regret definition:**
$$\text{Regret}(T) = T \cdot V^* - \sum_{t=1}^T V^{\pi_t}$$

**Lower bound** (any algorithm):
$$\text{Regret}(T) = \Omega(\sqrt{HSAT})$$

where $H$ = horizon, $S$ = states, $A$ = actions, $T$ = time

**Upper bound** (good algorithms like UCRL2):
$$\text{Regret}(T) = O(HS\sqrt{AT})$$

Nearly optimal!

### 8.3 Intrinsic Motivation Theory

**Learning progress** (Schmidhuber 1991):
$$r^{\text{int}} = |L_t - L_{t-1}|$$
where $L_t$ = prediction loss at time $t$

**Empowerment** (Klyubin et al. 2005):
$$E(s) = I(A; S'|s)$$
Mutual information between actions and future states

**Maximizes**: Agent's influence on environment

## Summary

### Key Takeaways

1. **Exploration-exploitation tradeoff** is fundamental to RL

2. **Multi-armed bandits** provide foundational algorithms:
   - ε-greedy: Simple but suboptimal
   - UCB: Optimal regret, count-based
   - Thompson Sampling: Bayesian, optimal

3. **Count-based exploration**: Bonus for novel states
   - Simple counts (tabular)
   - Pseudo-counts (high-dimensional)
   - Hash-based (practical)

4. **Curiosity-driven exploration**: Learn to be surprised
   - Forward dynamics error (prediction error)
   - ICM (feature space, inverse model)
   - RND (random network distillation)

5. **Information-theoretic**: Entropy, information gain
   - Maximize policy entropy (SAC)
   - Maximize state diversity
   - Bayesian information gain

6. **Modern methods**:
   - NGU: Episodic + lifelong novelty
   - Agent57: Population of policies
   - Go-Explore: Archive-based

7. **Method selection** depends on:
   - Reward density
   - Environment stochasticity
   - State space size
   - Computational budget

8. **Practical tips**:
   - Intrinsic reward scaling is critical
   - Combine multiple methods
   - Decay exploration over time

### Next Steps

- **Lesson 12b**: Implement count-based, ICM, RND exploration
- **Lesson 13**: Multi-agent RL
- **Lesson 14**: Hierarchical RL

### Further Reading

1. Sutton & Barto (2018) - Chapter 2: Multi-armed Bandits
2. Bellemare et al. (2016) - "Unifying Count-Based Exploration and Intrinsic Motivation"
3. Pathak et al. (2017) - "Curiosity-driven Exploration by Self-supervised Prediction" (ICM)
4. Burda et al. (2018) - "Exploration by Random Network Distillation" (RND)
5. Badia et al. (2020) - "Never Give Up: Learning Directed Exploration Strategies" (NGU)
6. Badia et al. (2020) - "Agent57: Outperforming the Atari Human Benchmark"
7. Ecoffet et al. (2021) - "First Return, Then Explore" (Go-Explore)

## Exercises

### Theoretical Questions

1. **Prove** that UCB1 achieves logarithmic regret for multi-armed bandits

2. **Derive** the pseudo-count formula from density models

3. **Analyze** why forward dynamics error fails in stochastic environments

4. **Explain** how ICM's inverse model filters out irrelevant features

5. **Compare** epistemic vs aleatoric uncertainty in exploration

### Computational Exercises

6. **Implement** UCB and Thompson Sampling for multi-armed bandits

7. **Compare** ε-greedy, UCB, Thompson Sampling on 10-armed testbed

8. **Build** count-based exploration for gridworld

9. **Implement** simple forward dynamics curiosity

10. **Compare** different intrinsic reward scalings (β values)

### Practical Applications

11. **Implement** ICM for Atari game with sparse rewards

12. **Build** RND exploration module

13. **Combine** count-based + ICM bonuses

14. **Analyze** exploration in Montezuma's Revenge with different methods

15. **Implement** decaying intrinsic reward coefficient (curriculum)